In [3]:
import torch
from torch import nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [5]:
df = pd.read_csv("/Users/animeshsingh/Desktop/Datasets/Fashion_MNIST/fashion-mnist_train.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.85, random_state = 25)

In [10]:
X_train = X_train / 255
X_test = X_test / 255

In [17]:
X_train[0].shape

(784,)

In [13]:
class Datass(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype = torch.float32).reshape(-1,1,28,28)
        self.y = torch.tensor(y, dtype = torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [14]:
train_data = Datass(X_train, y_train)
test_data = Datass(X_test, y_test)

In [16]:
train_load = DataLoader(train_data, batch_size = 128, shuffle = True)
test_load = DataLoader(test_data, batch_size = 128)

In [20]:
class CNN(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.X = nn.Sequential(
            nn.Conv2d(channels, 48, kernel_size = 3, padding = 'same'),
            nn.ReLU(),
            nn.BatchNorm2d(48),
            nn.MaxPool2d(kernel_size = 2, stride = 2),
            nn.Conv2d(48, 64, kernel_size = 3, padding = 'same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size = 2, stride = 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 256),
            nn.ReLU(),
            nn.Dropout(p = 0.2),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(p = 0.2),
            nn.Linear(64, 10)
        )

    def forward(self, X):
        z = self.X(X)
        z = self.classifier(z)
        return z    

In [28]:
lr = 0.01
epochs = 100

In [29]:
model = CNN(1)
crit = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = lr)

In [31]:
for epoch in range(epochs):
    epoch_loss = 0
    for batch_X, batch_y in train_load:
        outputs = model(batch_X)
        loss = crit(outputs, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss = epoch_loss + loss.item()

    avg_loss = epoch_loss / len(train_load)
    print(f'epoch: {epoch} loss: {avg_loss}')

epoch: 0 loss: 0.7729185144405317
epoch: 1 loss: 0.414302367278209
epoch: 2 loss: 0.34265304560350596
epoch: 3 loss: 0.3010876997371664
epoch: 4 loss: 0.2767533242030251
epoch: 5 loss: 0.2566402277962905
epoch: 6 loss: 0.23596420716074176
epoch: 7 loss: 0.22103018646028108
epoch: 8 loss: 0.2068446497981411
epoch: 9 loss: 0.20003486792545272
epoch: 10 loss: 0.1853073362195701
epoch: 11 loss: 0.17555735267717437
epoch: 12 loss: 0.16781761209692872
epoch: 13 loss: 0.1594999531157931
epoch: 14 loss: 0.15148824808143435
epoch: 15 loss: 0.14019544927734778
epoch: 16 loss: 0.13369795348597946
epoch: 17 loss: 0.12550819765094826
epoch: 18 loss: 0.11944646596964589
epoch: 19 loss: 0.11122936688195494
epoch: 20 loss: 0.10314776667657502
epoch: 21 loss: 0.09763846769088641
epoch: 22 loss: 0.09231380458669107
epoch: 23 loss: 0.08802139843559653
epoch: 24 loss: 0.07990887804997894
epoch: 25 loss: 0.07570047715776845
epoch: 26 loss: 0.07027598578286798
epoch: 27 loss: 0.06464040131403838
epoch: 28 l

In [32]:
model.eval()

CNN(
  (X): Sequential(
    (0): Conv2d(1, 48, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(48, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=256, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.2, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [33]:
tot = 0
correct = 0

with torch.no_grad():
    for batch_X, batch_y in test_load:
        outputs = model(batch_X)
        predicted = outputs.argmax(dim = 1)
        tot = tot + batch_y.shape[0]
        correct = correct + (predicted == batch_y).sum().item()

print(correct/tot)

0.9253333333333333
